# Objetivos de aprendizaje (ejercicio):
#### 1. Construir una estrategia concreta con opciones: long straddle periódico sobre SPY.
#### 2. Modificarla para obtener una versión delta-hedged (usando vuestras griegas) con el subyacente.
#### 3. Analizar el P&L histórico de ambas versiones.
#### 4. Simular envío de órdenes: como combo (straddle como una sola orden), y como patas sueltas (call y put por separado) analizando el riesgo de legging.
#### 5. Ver cómo se puede neutralizar Delta con otra opción y qué implica para Gamma, Vega y Theta.
#### 6. Reflexión: ¿qué habría cambiado usando SPX en lugar de SPY?

## 1. Construir una estrategia concreta con opciones: long straddle periódico sobre SPY.

Se utilizará un snapshot del Miércoles 7 de Enero, para las options de SPY con vencimiento el 06 de Febrero, ~30 días:

!["Options SPY"](resources/SPY_options_06022026.png)

In [ ]:
import pandas as pd

# Obtenemos los datos de la captura de pantalla del TWS
SPY_CURRENT_PRICE = 693.8
CONTRACT_MULTIPLIER = 100
df = pd.read_csv('resources/SPY_options_06022026.csv')

def long_straddle(current_price):
    # ATM -> menor diferencia con el precio actual
    df['diff'] = (df['Strike'] - current_price).abs()
    atm_row = df.loc[df['diff'].idxmin()]

    strike = atm_row['Strike']
    put_price = atm_row['Put_Ask']
    call_price = atm_row['Call_Ask']

    # Multiplicamos por CONTRACT_MULTIPLIER porque cada contrato controla CONTRACT_MULTIPLIER acciones
    coste_total = (call_price + put_price) * CONTRACT_MULTIPLIER

    breakeven_arriba = strike + (call_price + put_price)
    breakeven_abajo = strike - (call_price + put_price)

    print(f"Precio Snapshot: {current_price}")
    print(f"Strike seleccionado (ATM): {strike}")
    print(f"Coste de la operación: {coste_total} $")
    print(f"Ganamos si sube de: {breakeven_arriba}")
    print(f"Ganamos si baja de: {breakeven_abajo}")
    print(f"Delta inicial de la posición: {atm_row['Call_Delta'] + atm_row['Put_Delta']} (Call: {atm_row['Call_Delta']} | Put: {atm_row['Put_Delta']})")

    return coste_total

long_straddle(SPY_CURRENT_PRICE)

### Conclusión

En esta celda lo que estamos haciendo es comprar una call y una put y obtener el coste total

## 2. Modificarla para obtener una versión delta-hedged (usando vuestras griegas) con el subyacente.

In [ ]:
def long_straddle_delta_hedge(current_price):
    df['diff'] = (df['Strike'] - current_price).abs()
    atm_row = df.loc[df['diff'].idxmin()]

    strike = atm_row['Strike']
    put_price = atm_row['Put_Ask']
    call_price = atm_row['Call_Ask']
    put_delta = atm_row['Put_Delta']
    call_delta = atm_row['Call_Delta']

    # Coste total
    coste_total = (call_price + put_price) * CONTRACT_MULTIPLIER

    # Breakevens
    breakeven_arriba = strike + (call_price + put_price)
    breakeven_abajo = strike - (call_price + put_price)

    delta_total = call_delta + put_delta
    shares = -delta_total * CONTRACT_MULTIPLIER

    print(f"Precio Snapshot: {current_price}")
    print(f"Strike seleccionado (ATM): {strike}")
    print(f"Coste de la operación: {coste_total} $")
    print(f"Ganamos si sube de: {breakeven_arriba}")
    print(f"Ganamos si baja de: {breakeven_abajo}")
    print(f"Delta inicial de la posición: {delta_total} (Call: {atm_row['Call_Delta']} | Put: {atm_row['Put_Delta']})")
    if (shares > 0):
        print(f"Acciones a comprar para delta-hedge: {shares}")
    else:
        print(f"Acciones a vender para delta-hedge: {- shares}")

    return coste_total

long_straddle_delta_hedge(SPY_CURRENT_PRICE)

## 3. Analizar el P&L histórico de ambas versiones.

In [1]:
import numpy as np
from scipy.stats import norm
from datetime import datetime
import matplotlib.pyplot as plt

# Tasa de interes libre de riesgo. Puesto por defecto al valor que se facilitó en la parte de Aprendizaje de este notebook
r = 0.04
# Vollatilidad implicita. Se toma el valor aproximado de TWS
iv = 0.13
# Fecha de expiracion
expiration_date = datetime.strptime('06-02-26', '%d-%m-%y')

df_hist_prices = pd.read_csv('resources/SPY_hist_prices.csv')

def days_until_06022026(date_str):
    date = datetime.strptime(date_str, '%m-%d-%y')
    return (expiration_date - date).days

def black_scholes(S, K, T, sigma, option_type):
    if T <= 0:
        return 0, 0

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == 'call':
        price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
        delta = norm.cdf(d1)
    elif option_type == 'put':
        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
        delta = norm.cdf(d1) - 1
    else:
        return 0, 0

    return price, delta


def get_pnls(strike):
    pnl = {}

    s0 = df_hist_prices.iloc[0]['Close']
    t0 = days_until_06022026(df_hist_prices.iloc[0]['Date']) / 365

    # Simulamos el primer precio del historico
    c0_price, c0_delta = black_scholes(s0, strike, t0, iv, 'call')
    p0_price, p0_delta = black_scholes(s0, strike, t0, iv, 'put')

    delta_total_t0 = (c0_delta + p0_delta)
    coste_straddle_t0 = (c0_price + p0_price) * CONTRACT_MULTIPLIER

    st_prev = s0
    shares_total_pnl = 0
    shares_t_prev = -delta_total_t0 * CONTRACT_MULTIPLIER
    for _, row in df_hist_prices.iterrows():
        st = row['Close']
        date = row['Date']
        t = days_until_06022026(date) / 365

        # Calcular precios teoricos en t
        ct_price, ct_delta = black_scholes(st, strike, t, iv, 'call')
        pt_price, pt_delta = black_scholes(st, strike, t, iv, 'put')

        coste_straddle_tt = (ct_price + pt_price) * CONTRACT_MULTIPLIER

        # Cuanto he ganado/perdido frente a mi posicion inicial
        pnl_t = coste_straddle_tt - coste_straddle_t0

        # hedge
        # Cuando he ganado / perdido con las acciones que he comprado / vendido
        shares_total_pnl += shares_t_prev * (st - st_prev)
        # Con cuantas acciones me cubro (comprando o vendiendo) para T = t
        delta_total_t = ct_delta + pt_delta

        pnl[date] = {
            'basic': pnl_t,
            'hedged': pnl_t + shares_total_pnl
        }

        # Actualizamos para t (proximo t - 1)
        st_prev = st
        shares_t_prev = -delta_total_t * CONTRACT_MULTIPLIER

    return pnl

pnl = get_pnls(SPY_CURRENT_PRICE)
pnl_df = pd.DataFrame.from_dict(pnl, orient='index')
pnl_df.index = pd.to_datetime(pnl_df.index, format='%m-%d-%y')

plt.figure(figsize=(12,6))
plt.plot(pnl_df.index, pnl_df['basic'], label='P&L sin hedge', color='red')
plt.plot(pnl_df.index, pnl_df['hedged'], label='P&L Delta-Hedged', color='blue')

plt.title('P&L Histórico - Straddle con y sin Delta-Hedge')
plt.xlabel('Fecha')
plt.ylabel('P&L ($)')
plt.grid(True)
plt.legend()
plt.show()


NameError: name 'pd' is not defined

## 4. Simular envío de órdenes: como combo (straddle como una sola orden), y como patas sueltas (call y put por separado) analizando el riesgo de legging.

**NOTA**: Al ejecutar este código en un Jupiter Notebook se obtiene el error "*RuntimeError: Timeout should be used inside a task*". Para solventarlo, es necesario reiniciar el kernel y lanzar esta celda.

Para poder realizar las acciones con la cuenta demo (no tenemos acceso a los datos en tiempo real) se ha hecho BYPASS de las medidas de seguridad de la TWS:

![bypass](resources/bypass.png)

In [1]:
from ib_insync import *

# ATM
strike = 694
# Vencimiento de los ejemplos anteriores
expiration_date_str = '20260206'
spy = Stock('SPY', 'SMART', 'USD')

def straddle_atomic():

    put = Option('SPY', expiration_date_str, strike, 'P', 'SMART')
    call = Option('SPY', expiration_date_str, strike, 'C', 'SMART')

    # Se rellenan los ids en call y put
    ib.qualifyContracts(call, put)

    straddle_bag = Contract(
    symbol='SPY',
    secType='BAG',
    exchange='SMART',
    currency='USD',

    comboLegs=[
        ComboLeg(conId=call.conId, ratio=1, action='BUY'),
        ComboLeg(conId=put.conId, ratio=1, action='BUY')
        ]
    )

    trade = ib.placeOrder(straddle_bag, MarketOrder('BUY', 1, tif='GTC'))

    while not trade.isDone():
        ib.sleep(1)

    trade_price = trade.orderStatus.avgFillPrice
    print(f"Straddle como una sola orden: {trade_price}")
    return trade_price, call, put

def straddle_legged():

    put = Option('SPY', expiration_date_str, strike, 'P', 'SMART')
    call = Option('SPY', expiration_date_str, strike, 'C', 'SMART')

    # Se rellenan los ids en call y put
    ib.qualifyContracts(call, put)

    # Vendemos la Call
    trade_call = ib.placeOrder(call, MarketOrder('BUY', 1))
    while not trade_call.isDone():
        ib.sleep(0.1)

    # Esperamos 10 segundos
    ib.sleep(10)

    # Vendemos la Put
    trade_put = ib.placeOrder(put, MarketOrder('BUY', 1))
    while not trade_put.isDone():
        ib.sleep(0.1)

    trades_price = trade_call.orderStatus.avgFillPrice + trade_put.orderStatus.avgFillPrice
    print(f"Straddle como patas sueltas (Call+Put) = { trades_price }")
    return trades_price, call, put

# Necesario en notebooks (evita el "event loop is already running")
util.startLoop()

# Conectar a la API
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=12)
ib.reqMarketDataType(3)

# Combo
combo_price, combo_call, combo_put = straddle_atomic()
# Legged
legged_price, legged_call, legged_put = straddle_legged()

print(f"Precio al hacer 'combo': {combo_price}. Precio al hacer legged: {legged_price}")


API connection failed: ConnectionRefusedError(61, "Connect call failed ('127.0.0.1', 7497)")
Make sure API port on TWS/IBG is open


ConnectionRefusedError: [Errno 61] Connect call failed ('127.0.0.1', 7497)

### Conclusión

Este código solo es válido si el mercado está abierto. En caso contrario, se obtiene el error "*Error 10349, reqId 43: Order TIF was set to DAY based on order preset.*". Incluso si ponemos la opción para que el Market order sea "*GTC*" (Good Till Cancelled), no podemos ejecutarlo, puesto que no obtenemos un precio de la venta de las opciones. Esto se refleja en TWS de la siguiente forma:

![Pending](resources/pending.png)

No obstante, si comparamos el ejecutar la operación de forma atómica (vender call y put en la misma orden) estamos evitando el riesgo que supone que el precio subyacente varíe entre operaciones y, por tanto, el precio de la segunda orden que ejerzamos (call o put) tenga un precio inferior, resultando en un menor beneficio o, incluso, en una pérdida. Por tanto, realizar la operación de forma atómica lo que permite es neutralizar Delta.

## 5. Ver cómo se puede neutralizar Delta con otra opción y qué implica para Gamma, Vega y Theta.

Para neutralizar Delta debemos buscar opciones que contrarresten el Delta actual. Esto es, si tenemos Delta positiva, comprar una opción put o vender una call (añaden Delta negativa) o, si bien tenemos Delta negativa, vender una opción put ( - * - = +) o comprar una call (Delta positiva).

#### Gamma
Gamma es positiva para el comprador y mide cuánto se mueve Delta respecto al subyacente.
* Si compramos una opción para neutralizar Delta, Gamma es mayor porque ahora hay una Delta adicional que debe sumarse y que también es susceptible a los cambios del subyacente.
* Si vendemos una opción para neutralizar Delta, Gamma disminuye, ya que vender tiene signo negativo.
#### Vega
Vega es positiva para el comprador y mide cuánto cambia el precio de una opción con respecto a la volatilidad implícita.
* Si compramos una opción para neutralizar Delta, Vega aumenta porque se añade otra posibilidad de ganar dinero si la volatilidad aumenta.
* Si vendemos una opción para neutralizar Delta, Vega disminuye porque estamos menos expuestos a la volatilidad implícita.
#### Theta
Theta es negativo para el comprador y mide cuánto varía la prima con respecto al paso del tiempo.
* Si compramos una opción para neutralizar Delta, estamos añadiendo otra variable que caduca, por lo tanto, Theta es menor (más negativo), ya que la pérdida puede ser mayor según pasan los días.
* Si vendemos una opción para neutralizar Delta, quitamos una variable que caduca, por tanto, el valor Theta de nuestra cartera es mayor (menos negativo), ya que tenemos un riesgo menos.

**NOTA**: Este apartado necesita ejecutar el anterior para funcionar

In [ ]:
import pandas as pd
from ib_insync import *

def greeks(contracts):
    data = []

    tickers = ib.reqTickers(*contracts)
    # Anyadimos un tiempo prudente para que se generen los datos
    ib.sleep(5)

    for t in tickers:
        g = t.modelGreeks
        if g:
            data.append({
                'Contrato': f"{t.contract.symbol} {t.contract.right} {t.contract.strike}",
                'Delta': g.delta,
                'Gamma': g.gamma,
                'Vega': g.vega,
                'Theta': g.theta
            })

    df = pd.DataFrame(data)

    # Obtenemos el total
    return df[['Delta', 'Gamma', 'Vega', 'Theta']].sum()

def neutralize_delta(delta):
    print(f"Delta: {delta}")

    if abs(delta) > 0:

        action = ''
        option = Option('SPY', expiration_date_str, strike + 5, 'C', 'SMART')

        if delta > 0:
            # Vendemos call para restar Delta y restar Gamma
            print("Delta > 0 -> Vendemos call")
            action = 'SELL'

        else:
            # Compramos call para sumar Delta y sumar Gamma
            print("Delta negativa <= 0 -> Compramos call")
            action = 'BUY'

        ib.placeOrder(option, MarketOrder(action, 1))

        return option

    return None

# Recuperamos los datos del combo
options = [combo_call, combo_put]

combo_greek_values = greeks(options)
print(f"Valores antes de neutralizar: {combo_greek_values}")
# Neutralizamos
neutralizing_option = neutralize_delta(combo_greek_values['Delta'])
if neutralizing_option:
    options.append(neutralizing_option)
    # Obtenemos los nuevos valores
    neutralized_greek_values = greeks(neutralizing_option)
    print(f"Valores despues de neutralizar: {neutralized_greek_values}")

## 6. Reflexión: ¿qué habría cambiado usando SPX en lugar de SPY?

Si hubiéramos usado SPX (índice) en lugar de SPY (ETF) habríamos encontrado valores de Gamma,